In [ ]:
!pip install sentence-transformers faiss-cpu requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.6 MB/s eta 0:00:00


In [ ]:
text = """
Transformers use self-attention mechanisms.
RAG stands for Retrieval Augmented Generation.
Vector databases store embeddings and perform similarity search.
PCA reduces dimensionality by projecting onto principal components.
"""

with open("notes.txt", "w") as f:
    f.write(text)

print("File created.")


File created.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load embedding model (runs locally inside Colab)
model = SentenceTransformer("all-MiniLM-L6-v2")

# Read file
with open("notes.txt", "r") as f:
    text = f.read()

# Split into chunks (simple version)
chunks = text.strip().split("\n")

# Convert text to embeddings
embeddings = model.encode(chunks)

# Convert to numpy float32
embeddings = np.array(embeddings).astype("float32")

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Embeddings created and stored in vector database.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings created and stored in vector database.


In [ ]:
def retrieve(query, top_k=2):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return [chunks[i] for i in indices[0]]

# Test it
print(retrieve("What is RAG?"))


['RAG stands for Retrieval Augmented Generation.', 'Vector databases store embeddings and perform similarity search.']


In [ ]:
import requests

HF_TOKEN = "hf_***********************"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "model": "meta-llama/Llama-3.1-8B-Instruct",
    "messages": [
        {
            "role": "user",
            "content": "How many G's are in huggingface?"
        }
    ],
    "max_tokens": 50
}


response = requests.post(
    "https://router.huggingface.co/v1/chat/completions",
    headers=headers,
    json=payload
)

print("Status:", response.status_code)
print(response.text)


In [ ]:
import requests

HF_TOKEN = "hf_***********************"  # paste again locally (not publicly)

API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json"
}




def generate_answer(query):
    payload = {
        "model": "meta-llama/Llama-3.1-8B-Instruct",
        "messages": [
            {"role": "system", "content": "You are an AI assistant specialized in machine learning and AI."}, # to make it smarter and to direct its function i guess
            {"role": "user", "content": query}
        ],
        "max_tokens": 150
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code != 200:
        return f"Error: {response.text}"

    return response.json()["choices"][0]["message"]["content"]


print(generate_answer("Explain RAG in simple terms"))


RAG stands for Relation-aware Graph Network. It's a type of neural network architecture used in artificial intelligence, particularly in graph-based tasks.

In simple terms, RAG is a neural network that tries to understand relationships (relations) between different objects (entities) in a graph. A graph is like a map that shows how different things are connected to each other.

Here's a simple analogy:

* Entities are like people in a company.
* Relations are like "works for" or "manages" between people.
* A graph is like a map of the company where people are connected to each other through relationships.

RAG tries to identify these relationships and learn from them, which helps to make predictions, classifications, or recommendations.

For example,


In [ ]:
print(generate_answer(
    "In the field of machine learning, deep learning, and artificial intelligence, what is Retrieval Augmented Generation (RAG)? Explain"
))


Retrieval Augmented Generation (RAG) is a technique in the field of natural language processing (NLP) and artificial intelligence (AI), which combines two main components: retrieval and generation. It has gained significant attention in recent years due to its ability to create coherent and contextually relevant texts, even for complex and long-tail tasks.

**Components of RAG:**

1. **Retrieval:** In RAG, a retrieval component first searches for relevant knowledge from a given dataset or knowledge graph. This knowledge is typically represented as a set of short text snippets or passages that are relevant to the task at hand. The goal of the retrieval component is to find the most informative and relevant pieces of knowledge that are likely to be useful
